# Incremental Models

## Overview
An **incremental model** builds on the existing table rather than rebuilding from scratch. Instead of `DROP + CREATE TABLE`, dbt inserts or merges only new or changed rows.

**Why incremental models exist:**
A `fct_page_views` table with 5 billion rows takes hours to rebuild from scratch. An incremental run processing only the last hour of new events takes seconds.

**The `is_incremental()` pattern:**
```sql
{% if is_incremental() %}
  WHERE viewed_at > (SELECT MAX(viewed_at) FROM {{ this }})
{% endif %}
```
- `is_incremental()` returns `True` on runs after the target table already exists
- `{{ this }}` compiles to the current model's fully-qualified table name
- On first run or `--full-refresh`, the block is skipped and all rows are loaded

**Incremental lifecycle:**
```
First run:    CREATE TABLE fct_page_views AS (SELECT ...)
dbt run:      INSERT ... WHERE viewed_at > MAX - buffer
dbt run:      INSERT ... WHERE viewed_at > MAX - buffer
Schema change → dbt run --full-refresh -s fct_page_views → DROP + recreate
```

---

In [1]:
model_sql = (
    "-- models/marts/finance/fct_page_views.sql\n"
    "{{ config(\n"
    "    materialized  = 'incremental',\n"
    "    unique_key    = 'page_view_id',\n"
    "    on_schema_change = 'append_new_columns'\n"
    ") }}\n\n"
    "SELECT page_view_id, session_id, user_id, page_url,\n"
    "       viewed_at, duration_seconds,\n"
    "       CURRENT_TIMESTAMP  AS dbt_loaded_at\n"
    "FROM {{ source('analytics', 'raw_page_views') }}\n\n"
    "{% if is_incremental() %}\n"
    "  WHERE viewed_at > (\n"
    "      SELECT MAX(viewed_at) - INTERVAL '1 hour'\n"
    "      FROM {{ this }}\n"
    "  )\n"
    "{% endif %}"
)
print("=== Incremental model: fct_page_views.sql ===")
print(model_sql)
print()
lifecycle = (
    "First run:\n"
    "  CREATE TABLE fct_page_views AS (SELECT ... FROM raw_page_views)\n\n"
    "Subsequent runs (is_incremental() = True):\n"
    "  CREATE TEMP TABLE fct_page_views__dbt_tmp AS (\n"
    "      SELECT ... WHERE viewed_at > MAX(viewed_at) - INTERVAL '1 hour'\n"
    "  );\n"
    "  -- Merge temp table into fct_page_views\n\n"
    "Full rebuild:\n"
    "  dbt run --full-refresh -s fct_page_views\n"
    "  -- Drops fct_page_views and rebuilds from scratch"
)
print("What dbt executes:")
print(lifecycle)

=== Incremental model: fct_page_views.sql ===
-- models/marts/finance/fct_page_views.sql
{{ config(
    materialized  = 'incremental',
    unique_key    = 'page_view_id',
    on_schema_change = 'append_new_columns'
) }}

SELECT page_view_id, session_id, user_id, page_url,
       viewed_at, duration_seconds,
       CURRENT_TIMESTAMP  AS dbt_loaded_at
FROM {{ source('analytics', 'raw_page_views') }}

{% if is_incremental() %}
  WHERE viewed_at > (
      SELECT MAX(viewed_at) - INTERVAL '1 hour'
      FROM {{ this }}
  )
{% endif %}

What dbt executes:
First run:
  CREATE TABLE fct_page_views AS (SELECT ... FROM raw_page_views)

Subsequent runs (is_incremental() = True):
  CREATE TEMP TABLE fct_page_views__dbt_tmp AS (
      SELECT ... WHERE viewed_at > MAX(viewed_at) - INTERVAL '1 hour'
  );
  -- Merge temp table into fct_page_views

Full rebuild:
  dbt run --full-refresh -s fct_page_views
  -- Drops fct_page_views and rebuilds from scratch


---
## Incremental strategies: append, merge, delete+insert

In [2]:
strategies = [
    ("append",
     "INSERT new rows; never update existing rows",
     "Immutable event logs. Fast. No unique_key needed."),
    ("merge (PostgreSQL default)",
     "UPDATE matching rows; INSERT new rows (UPSERT)",
     "Orders that change status. Requires unique_key."),
    ("delete+insert",
     "DELETE matching rows then INSERT",
     "BigQuery, Snowflake. Cleaner for some engines. Requires unique_key."),
]
print("=== Incremental strategies ===")
for name, behaviour, use_case in strategies:
    print(f"  Strategy:  {name}")
    print(f"  Behaviour: {behaviour}")
    print(f"  Use case:  {use_case}")
    print()

merge_example = (
    "{{ config(\n"
    "    materialized='incremental',\n"
    "    incremental_strategy='merge',\n"
    "    unique_key='order_id'\n"
    ") }}\n"
    "SELECT order_id, status, updated_at\n"
    "FROM {{ source('app', 'raw_orders') }}\n"
    "{% if is_incremental() %}\n"
    "WHERE updated_at > (SELECT MAX(updated_at) - INTERVAL '1 hour' FROM {{ this }})\n"
    "{% endif %}"
)
print("Merge strategy example:")
print(merge_example)

=== Incremental strategies ===
  Strategy:  append
  Behaviour: INSERT new rows; never update existing rows
  Use case:  Immutable event logs. Fast. No unique_key needed.

  Strategy:  merge (PostgreSQL default)
  Behaviour: UPDATE matching rows; INSERT new rows (UPSERT)
  Use case:  Orders that change status. Requires unique_key.

  Strategy:  delete+insert
  Behaviour: DELETE matching rows then INSERT
  Use case:  BigQuery, Snowflake. Cleaner for some engines. Requires unique_key.

Merge strategy example:
{{ config(
    materialized='incremental',
    incremental_strategy='merge',
    unique_key='order_id'
) }}
SELECT order_id, status, updated_at
FROM {{ source('app', 'raw_orders') }}
{% if is_incremental() %}
WHERE updated_at > (SELECT MAX(updated_at) - INTERVAL '1 hour' FROM {{ this }})
{% endif %}


---
## Late-arriving data and lookback windows

In [3]:
print("=== Late-arriving data and lookback windows ===")
late_model = (
    "{{ config(materialized='incremental', unique_key='event_id',\n"
    "          incremental_strategy='merge') }}\n\n"
    "-- Events can arrive up to 72 hours late from mobile clients.\n"
    "-- Reprocess the last 3 days on every run to catch late arrivals.\n"
    "SELECT event_id, user_id, event_type, occurred_at\n"
    "FROM {{ source('app', 'raw_events') }}\n"
    "{% if is_incremental() %}\n"
    "WHERE occurred_at >= NOW() - INTERVAL '72 hours'\n"
    "{% endif %}"
)
print(late_model)
print()
rows = [
    ("Wider lookback window", "Catches more late arrivals",  "More rows per run; slower"),
    ("append strategy",       "Fastest incremental run",     "Duplicates for late-arriving rows"),
    ("merge + unique_key",    "Handles late updates",        "Slower; needs MERGE support"),
    ("full-refresh daily",    "Always correct",              "Very slow for large tables"),
]
print(f"  {chr(34)}Approach{chr(34):<20s}  {chr(34)}Pro{chr(34):<28s}  Con")
print("  " + "-"*70)
for approach, pro, con in rows:
    print(f"  {approach:<28s}  {pro:<30s}  {con}")

=== Late-arriving data and lookback windows ===
{{ config(materialized='incremental', unique_key='event_id',
          incremental_strategy='merge') }}

-- Events can arrive up to 72 hours late from mobile clients.
-- Reprocess the last 3 days on every run to catch late arrivals.
SELECT event_id, user_id, event_type, occurred_at
FROM {{ source('app', 'raw_events') }}
{% if is_incremental() %}
WHERE occurred_at >= NOW() - INTERVAL '72 hours'
{% endif %}

  "Approach"                     "Pro"                             Con
  ----------------------------------------------------------------------
  Wider lookback window         Catches more late arrivals      More rows per run; slower
  append strategy               Fastest incremental run         Duplicates for late-arriving rows
  merge + unique_key            Handles late updates            Slower; needs MERGE support
  full-refresh daily            Always correct                  Very slow for large tables


---
## Common Pitfalls

**1. Forgetting `--full-refresh` after changing the model's column list**
The incremental INSERT/MERGE only adds rows to the existing table schema. Adding a new column requires `dbt run --full-refresh -s <model>` once to rebuild the table with the updated schema.

**2. Using `append` when rows can be updated**
An order moving through placed to shipped to completed appears three times in an append-only table. If rows can change after initial insertion, use `merge` with a `unique_key`.

**3. No late-arrival buffer in the WHERE clause**
An exact cutoff on MAX timestamp means any event that arrives even 1 second late is missed forever. Subtract a buffer such as `MAX(viewed_at) - INTERVAL "1 hour"` to reprocess a recent window.

**4. Running `--full-refresh` on large tables without a maintenance window**
A full-refresh on a 5-billion-row fact table drops and rebuilds the entire table, blocking downstream models and dashboards for hours. Treat it as a breaking change and schedule it outside business hours.

**5. `unique_key` not actually unique in the source**
If the source produces two rows with the same `unique_key` in one incremental batch, the merge strategy's result is non-deterministic. Add `unique` and `not_null` tests on `unique_key` and verify upstream deduplication.

**6. Using `this` outside an `is_incremental()` block**
On the very first run the target table does not yet exist. A subquery against `this` outside an `is_incremental()` guard fails on the first build. Always wrap `this` references inside the guard.


---
*sql_methods_library - Samantha McGarrigle*